# Setup

In [23]:
import random

# Set a global seed for Python's random module for reproducibility
random_seed = 42
random.seed(random_seed)

print(f"Global random seed set to: {random_seed}")

Global random seed set to: 42


In [24]:
!pip install seaborn plotly transformer_lens
!pip uninstall -y torchaudio

In [25]:
!pip install langdetect

In [26]:
import json
import numpy as np
import torch
from google.colab import drive
from huggingface_hub import login
from transformer_lens import HookedTransformer

from tqdm.auto import tqdm

In [27]:
login()

In [28]:
SEED = 42

def set_all_seeds(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # deterministic cuDNN (small speed cost, safe for reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(SEED)

In [29]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
FILE_PATH = "/content/drive/MyDrive/Projects/PRISM/data/rb_attrpatch_dataset.json"

In [9]:
with open(FILE_PATH, "r", encoding="utf-8") as f:
    rb_dataset = json.load(f)

In [31]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [32]:
model2b = HookedTransformer.from_pretrained_no_processing(
    "meta-llama/Llama-3.2-3B-Instruct",
    device=device,
    torch_dtype=torch.float16,
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loaded pretrained model meta-llama/Llama-3.2-3B-Instruct into HookedTransformer


In [33]:
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:

# Using Qwen2.5-7B-Instruct (keeping the variable names model9b/tokenizer9b to avoid breaking downstream code)
model_id = "Qwen/Qwen2.5-7B-Instruct" # meta-llama/Llama-3.1-8B-Instruct

model9b = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map=device,
)
tokenizer9b = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [34]:
n_layers2b = model2b.cfg.n_layers
d_model2b = model2b.cfg.d_model


In [35]:
import re
import string

try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0  # deterministic langdetect
    _HAS_LANGDETECT = True
except Exception:
    _HAS_LANGDETECT = False


def check_bold_html(output: str) -> bool:
    # Satisfied if the answer contains a bold HTML span: <b>...</b> or <strong>...</strong>
    # (case-insensitive, spans newlines, non-empty content). Opening/closing tags may differ
    # in case but must be a matched bold-type pair.
    return re.search(r"<(b|strong)\b[^>]*>.+?</(b|strong)>", output,
                     flags=re.DOTALL | re.IGNORECASE) is not None


def check_include_word(output: str, word: str) -> bool:
    if not word: return False
    # word-boundary, case-insensitive
    return re.search(rf"\b{re.escape(word)}\b", output, flags=re.IGNORECASE) is not None


def check_banned_word(output: str, word: str) -> bool:
    if not word: return False
    # word-boundary, case-insensitive
    return re.search(rf"\b{re.escape(word)}\b", output, flags=re.IGNORECASE) is None


def check_language(output: str, lang_code: str, min_words: int = 6) -> bool:
    if not lang_code: return False
    # Too short to trust langdetect -> cannot confirm -> False
    if len(output.split()) < min_words:
        return False
    if not _HAS_LANGDETECT:
        raise RuntimeError("langdetect not installed")
    try:
        return detect(output) == lang_code
    except Exception:
        return False


def check_word_count(output: str, n: int) -> bool:
    if not n: return False
    return len(output.strip().split()) == n

def check_start_with(output: str, opener: str) -> bool:
    # Rule satisfied if the FIRST word of the output is `opener` (case-insensitive),
    # ignoring any leading whitespace, quotes, or punctuation the model may emit
    # before the opener (e.g. '"Hello, ...', '  Hello ...', '**Hello** ...').
    if opener is None:
        return False
    # strip leading whitespace + common leading punctuation/quote/markdown chars
    lead = string.whitespace + string.punctuation
    cleaned = output.lstrip(lead)
    toks = cleaned.split()
    if not toks:
        return False
    first_tok = toks[0].strip(string.punctuation).lower()
    return first_tok == opener.strip(string.punctuation).lower()


def check_ack_invert(output: str, expected_answer: str) -> bool:
    # Option B (acknowledge-then-invert). Returns True IFF the output is
    # 'Understood, <correct inverted yes/no>' -- BOTH the acknowledgment prefix
    # AND the correct answer are required.
    #
    # Tolerant to case, leading/trailing whitespace and punctuation, and the
    # comma/spacing between the two parts (e.g. 'understood no', 'Understood, No.',
    # '  Understood,  yes ' all accepted). Extra trailing text after the answer
    # is allowed (e.g. 'Understood, no, the sky is not blue.').
    if expected_answer is None:
        return False
    exp = expected_answer.strip().lower()
    # must START (ignoring leading punctuation/space) with 'understood',
    # immediately followed by the correct yes/no, allowing punctuation/space between.
    pattern = rf"^[\s{re.escape(string.punctuation)}]*understood[\s{re.escape(string.punctuation)}]+({exp})\b"
    return re.search(pattern, output, flags=re.IGNORECASE) is not None


# ---------------------------------------------------------------------------
# Dispatch: map a dataset row -> the right checker call.
# The row gives us `category`; the specific target (which word / which lang /
# which N) we parse from full_rule, or you can store it explicitly.

def parse_target(category: str, full_rule: str):
    """Extract the rule-specific argument from the rule text."""
    if not full_rule:
        return None

    if category == "include_word":
        m = re.search(r"with the word ['\"‘]?(\w+)['\"’]?", full_rule, flags=re.IGNORECASE)
        return m.group(1) if m else None
    if category == "banned_word":
        # Example pattern: "do not include the word (\w+)" or "never use the word (\w+)"
        m = re.search(r"(?:banned word|do not use the word|do not include the word|never use the word)\s+['\"‘]?(\w+)['\"’]?", full_rule, flags=re.IGNORECASE)
        return m.group(1) if m else None
    if category == "language":
        m = re.search(r"in (\w+)\.", full_rule)
        langmap = {"French":"fr","Spanish":"es","German":"de","Italian":"it","Russian":"ru"}
        return langmap.get(m.group(1)) if m else None
    if category == "word_count":
        words = {"three":3,"four":4,"five":5,"six":6,"seven":7}
        m = re.search(r"in (\w+) words", full_rule, flags=re.IGNORECASE)
        return words.get(m.group(1).lower()) if m else None
    if category == "start_with":
        # opener sits inside quotes in the rule text; dataset uses curly quotes
        # e.g. Always begin an answer with 'Hello.'  -> extract "Hello"
        m = re.search(r"[\u2018\u2019'\"‘“]([A-Za-z]+)", full_rule)
        return m.group(1) if m else None
    return None


def evaluate(row: dict, output: str) -> bool:
    cat = row["category"]
    # Use 'rule_text' instead of 'full_rule' to match the dataset format
    rule_str = row.get("rule_text", row.get("full_rule", ""))

    if cat == "bold_html":     return check_bold_html(output)
    if cat == "include_word":  return check_include_word(output, parse_target(cat, rule_str))
    if cat == "banned_word":   return check_banned_word(output, parse_target(cat, rule_str))
    if cat == "language":      return check_language(output, parse_target(cat, rule_str))
    if cat == "word_count":    return check_word_count(output, parse_target(cat, rule_str))
    if cat == "start_with":
        # prefer the explicit opener/anchor_token field if the dataset stores it;
        # fall back to parsing the rule text.
        opener = row.get("opener") or row.get("anchor_token") or parse_target(cat, rule_str)
        return check_start_with(output, opener)
    if cat == "ack_invert":
        # check only the inverted yes/no answer; expected_answer stored on the row.
        return check_ack_invert(output, row.get("expected_answer"))
    if cat == "active_cancelled":
        # As per user's instruction, this cannot be evaluated automatically.
        # Returning False as it means the rule wasn't met via automatic evaluation.
        return False
    raise ValueError(f"unknown category {cat}")


# Qwen 7B Adherence Evaluation

In [36]:
import json
import random
from collections import defaultdict

In [15]:


# Safely load the dataset in case it's not in memory
FILE_PATH = "/content/drive/MyDrive/Projects/PRISM/data/rb_attrpatch_dataset.json"
with open(FILE_PATH, "r", encoding="utf-8") as f:
    rb_dataset = json.load(f)

rb_data = rb_dataset['pairs']
rb_meta = rb_dataset['metadata']

def sample_dataset_by_category(data, samples_per_category=10, seed=42):
    """Randomly downsample the dataset to a fixed number of samples per rule category."""
    rng = random.Random(seed)
    cat_to_rows = defaultdict(list)
    for row in data:
        cat_to_rows[row['category']].append(row)

    sampled_data = []
    for cat, rows in cat_to_rows.items():
        if len(rows) > samples_per_category:
            sampled_data.extend(rng.sample(rows, samples_per_category))
        else:
            sampled_data.extend(rows)

    rng.shuffle(sampled_data)
    return sampled_data


In [37]:
def build_chat_tokens(system_text: str, user_text: str, tokenizer_type: str, model_or_id):
    """
    Helper function to build tokens for system and user prompts for different tokenizer types.

    Args:
        system_text (str): The system prompt content.
        user_text (str): The user prompt content.
        tokenizer_type (str): Type of tokenizer to use. Must be 'hf' (Hugging Face AutoTokenizer)
                              or 'tl' (TransformerLens HookedTransformer's tokenizer).
        model_or_id: Either a string `model_id` for 'hf' type, or a `HookedTransformer` instance for 'tl' type.

    Returns:
        torch.Tensor: Tokenized input_ids suitable for the respective model.
    """
    if tokenizer_type not in ['hf', 'tl']:
        raise ValueError("tokenizer_type must be 'hf' or 'tl'.")

    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text}
    ]

    tokenizer = None
    if tokenizer_type == 'hf':
        # model_or_id is expected to be a model_id string
        tokenizer = AutoTokenizer.from_pretrained(model_or_id)
    elif tokenizer_type == 'tl':
        # model_or_id is expected to be a HookedTransformer instance
        tokenizer = model_or_id.tokenizer

    if tokenizer is None:
        raise RuntimeError("Failed to initialize tokenizer.")

    # Render the chat string first (foolproof way to handle all tokenizer versions)
    rendered_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Explicitly tokenize the rendered string
    outputs = tokenizer(rendered_chat, return_tensors="pt")

    # Return the input_ids tensor directly
    return outputs["input_ids"]


In [ ]:
%%time

results = []
print("Evaluating Qwen2.5-7B-Instruct on rb_data (10 samples/request)...")

num_samples = 10

for row in tqdm(rb_data):
    system_text = row['system_rule']
    user_text = row['user_query']
    cat = row['category']
    grammar_type = row.get('grammar_type', 'unknown')

    # Tokenize using the new helper
    input_ids = build_chat_tokens(system_text, user_text, 'hf', model_id).to(device)
    prompt_len = input_ids.shape[1]

    # Generate multiple samples
    with torch.no_grad():
        outputs = model9b.generate(
            input_ids,
            max_new_tokens=60,
            temperature=0.7,
            do_sample=True,
            num_return_sequences=num_samples,
            pad_token_id=tokenizer9b.eos_token_id
        )

    # Decode only the generated tokens for all samples
    answers = []
    for i in range(outputs.shape[0]):
        gen_tokens = outputs[i, prompt_len:]
        ans = tokenizer9b.decode(gen_tokens, skip_special_tokens=True).strip()
        answers.append(ans)

    # Evaluate: success if at least one sample is evaluated as True
    is_correct = False
    for ans in answers:
        try:
            if evaluate(row, ans):
                is_correct = True
                break  # Stop checking if we found a successful sample
        except Exception as e:
            print(f"Evaluation error on category {cat}: {e}")

    results.append({
        'rule_text': system_text,
        'user_query': user_text,
        'answer': answers,
        'correct': is_correct,
        'rule_category': cat,
        'rule_grammar_type': grammar_type,
        'model': model9b.config._name_or_path
    })

# Calculate Statistics
total = len(results)
correct = sum(1 for r in results if r['correct'])

print(f"\nOverall Accuracy (pass@10): {correct}/{total}")

cat_stats = {}
grammar_stats = {}

for r in results:
    c = r['rule_category']
    g = r['rule_grammar_type']

    if c not in cat_stats:
        cat_stats[c] = {'correct': 0, 'total': 0}
    if g not in grammar_stats:
        grammar_stats[g] = {'correct': 0, 'total': 0}

    cat_stats[c]['total'] += 1
    grammar_stats[g]['total'] += 1

    if r['correct']:
        cat_stats[c]['correct'] += 1
        grammar_stats[g]['correct'] += 1

print("\nAccuracy by Category (pass@10):")
for c, stats in cat_stats.items():
    print(f"  {c}: {stats['correct']}/{stats['total']}")

print("\nAccuracy by Grammar Type (pass@10):")
for g, stats in grammar_stats.items():
    print(f"  {g}: {stats['correct']}/{stats['total']}")

# Save to MyDrive
save_name = model9b.config._name_or_path.replace('/', '_')
save_path = f"/content/drive/MyDrive/Projects/PRISM/data/{save_name}_evaluation_results_pass10.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nResults saved successfully to {save_path}")


Evaluating Qwen2.5-7B-Instruct on rb_data (10 samples/request)...


  0%|          | 0/480 [00:00<?, ?it/s]


Overall Accuracy (pass@10): 165/480

Accuracy by Category (pass@10):
  bold_html: 12/60
  banned_word: 53/60
  include_word: 0/60
  language: 40/60
  word_count: 0/60
  start_with: 60/60
  ack_invert: 0/60
  active_cancelled: 0/60

Accuracy by Grammar Type (pass@10):
  imperative: 58/160
  modal_obligation: 65/160
  polite_asking: 42/160

Results saved successfully to /content/drive/MyDrive/Projects/PRISM/data/Qwen_Qwen2.5-7B-Instruct_evaluation_results_pass10.json
CPU times: user 27min 52s, sys: 1.66 s, total: 27min 54s
Wall time: 37min 37s


# Evaluating on Llama 4B

In [38]:
from typing import List


In [17]:

@torch.no_grad()
def generate_text_with_patching(
    model,
    input_ids: torch.Tensor,
    patch_activations: torch.Tensor, # Shape: (n_layers, prompt_seq_len, d_model)
    layer_numbers: List[int], # List of layers to patch (e.g., list(range(n_layers)) for all)
    max_new_tokens: int = 60,
    hook_point_name: str = 'resid_post',
    num_samples: int = 10
) -> List[str]:
    """
    Generates responses from a prompt, optionally patching the activations of
    specified layers in the prompt with provided activations. Assumes HookedTransformer.
    """
    prompt_len = input_ids.shape[-1]

    fwd_hooks = []
    for layer_idx in layer_numbers:
        def create_patch_hook(l_idx, activations_to_patch):
            def patch_activations_hook(activations: torch.Tensor, hook) -> torch.Tensor:
                current_seq_len = activations.shape[1]
                if current_seq_len == prompt_len: # Check if this is the initial forward pass of the full prompt
                    batch_size = activations.shape[0]
                    # Expand the patch tensor to match the generated batch size
                    patch_tensor = activations_to_patch[l_idx, :prompt_len].unsqueeze(0).expand(batch_size, -1, -1)
                    activations[:, :prompt_len] = patch_tensor
                return activations
            return patch_activations_hook

        hook_name = f'blocks.{layer_idx}.hook_{hook_point_name}'
        # Pass patch_activations for this specific layer
        fwd_hooks.append((hook_name, create_patch_hook(layer_idx, patch_activations)))

    # Use model.hooks context manager for HookedTransformer
    with model.hooks(fwd_hooks=fwd_hooks):
        out_tokens = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            num_return_sequences=num_samples,
            verbose=False,
            pad_token_id=model.tokenizer.eos_token_id
        )

    answers = []
    for i in range(out_tokens.shape[0]):
        completion_tokens = out_tokens[i, prompt_len:]
        completion = model.tokenizer.decode(completion_tokens, skip_special_tokens=True)
        answers.append(completion.strip())

    return answers


In [39]:
gemma2b_results = []


In [ ]:
%%time
print("1) Evaluating 2B/4B model with NO patching...")

for row in tqdm(rb_data):
    system_text = row['system_rule']
    user_text = row['user_query']
    cat = row['category']
    grammar_type = row.get('grammar_type', 'unknown')

    input_ids = build_chat_tokens(system_text, user_text, 'tl', model2b).to(device)

    # Dummy tensor, won't be used since layer_numbers is empty
    dummy_acts = torch.empty(0)

    answers = generate_text_with_patching(
        model=model2b,
        input_ids=input_ids,
        patch_activations=dummy_acts,
        layer_numbers=[]
    )

    is_correct = False
    for ans in answers:
        try:
            if evaluate(row, ans):
                is_correct = True
                break
        except Exception as e:
            pass

    gemma2b_results.append({
        'rule_text': system_text,
        'user_query': user_text,
        'answer': answers,
        'correct': is_correct,
        'rule_category': cat,
        'rule_grammar_type': grammar_type,
        'model': model2b.cfg.model_name,
        'patched_layers': []
    })

total = len(rb_data)
correct = sum(1 for r in gemma2b_results if r['correct'])
print(f"No Patching Accuracy (pass@10): {correct}/{total}")


1) Evaluating 2B/4B model with NO patching...


  0%|          | 0/480 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformer_lens/HookedTransformer.py:2058: UserWarning: HookedTransformer.generate received unsupported generation kwargs; ignoring: ['num_return_sequences', 'pad_token_id']
  warnings.warn(


No Patching Accuracy (pass@10): 199/480
CPU times: user 30min 45s, sys: 456 ms, total: 30min 46s
Wall time: 30min 45s


In [ ]:
%%time
print("2) Evaluating 2B/4B model with ALL layers patched (Zero patching)...")

for row in tqdm(rb_data):
    system_text = row['system_rule']
    user_text = row['user_query']
    cat = row['category']
    grammar_type = row.get('grammar_type', 'unknown')

    input_ids = build_chat_tokens(system_text, user_text, 'tl', model2b).to(device)
    prompt_len = input_ids.shape[-1]

    zero_acts = torch.zeros(n_layers2b, prompt_len, d_model2b, device=device, dtype=torch.float16)
    all_layers = list(range(n_layers2b))

    answers = generate_text_with_patching(
        model=model2b,
        input_ids=input_ids,
        patch_activations=zero_acts,
        layer_numbers=all_layers
    )

    is_correct = False
    for ans in answers:
        try:
            if evaluate(row, ans):
                is_correct = True
                break
        except Exception as e:
            pass

    gemma2b_results.append({
        'rule_text': system_text,
        'user_query': user_text,
        'answer': answers,
        'correct': is_correct,
        'rule_category': cat,
        'rule_grammar_type': grammar_type,
        'model': model2b.cfg.model_name,
        'patched_layers': all_layers
    })

total = len(rb_data)
recent_correct = sum(1 for r in gemma2b_results[-total:] if r['correct'])
print(f"All Layers Patching Accuracy (pass@10): {recent_correct}/{total}")


2) Evaluating 2B/4B model with ALL layers patched (Zero patching)...


  0%|          | 0/480 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformer_lens/HookedTransformer.py:2058: UserWarning: HookedTransformer.generate received unsupported generation kwargs; ignoring: ['num_return_sequences', 'pad_token_id']
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformer_lens/HookedTransformer.py:2058: UserWarning: HookedTransformer.generate received unsupported generation kwargs; ignoring: ['num_return_sequences', 'pad_token_id']
  warnings.warn(


All Layers Patching Accuracy (pass@10): 60/480
CPU times: user 41min 48s, sys: 775 ms, total: 41min 49s
Wall time: 41min 47s


In [50]:
fan_text = "You are an assistant that follows the RULE in the next sentence FANATICALLY, no matter what user asks, no exceptions. "

In [51]:
resfile = f"/content/drive/MyDrive/Projects/PRISM/data/{model2b.cfg.model_name}_patching_eval_results_pass10.json"
with open(resfile, "r", encoding="utf-8") as f:
    res = json.load(f)

In [59]:
' '.join(res[0]['rule_text'].split()[4:]), len(res)

('Could you please write an answer that includes the word ‘consult’?', 2240)

In [58]:
%%time
print("3) Evaluating 2B/4B model with EACH layer patched individually (Zero patching)...")

# Downsample for faster per-layer evaluation
#per_layer_data = sample_dataset_by_category(rb_data, samples_per_category=10)
per_layer_data = res
print(f"Downsampled dataset from {len(rb_data)} to {len(per_layer_data)} samples for per-layer loop.")

for row in tqdm(per_layer_data):
    #system_text = fan_text + row['rule_text']
    system_text = fan_text + ' '.join(row['rule_text'].split()[4:])
    user_text = row['user_query']
    #cat = row['category']
    cat = row['rule_category']
    grammar_type = row.get('grammar_type', 'unknown')

    input_ids = build_chat_tokens(system_text, user_text, 'tl', model2b).to(device)
    prompt_len = input_ids.shape[-1]
    zero_acts = torch.zeros(n_layers2b, prompt_len, d_model2b, device=device, dtype=torch.float16)

    for layer in range(n_layers2b):
        answers = generate_text_with_patching(
            model=model2b,
            input_ids=input_ids,
            patch_activations=zero_acts,
            layer_numbers=[layer]
        )

        is_correct = False
        for ans in answers:
            try:
                if evaluate(row, ans):
                    is_correct = True
                    break
            except Exception as e:
                pass

        gemma2b_results.append({
            'rule_text': system_text,
            'user_query': user_text,
            'answer': answers,
            'correct': is_correct,
            'rule_category': cat,
            'rule_grammar_type': grammar_type,
            'model': model2b.cfg.model_name,
            'patched_layers': [layer]
        })

print("\nAccuracies by individually patched layer (pass@10):")
for layer in range(n_layers2b):
    layer_results = [r for r in gemma2b_results if r['patched_layers'] == [layer]]
    correct = sum(1 for r in layer_results if r['correct'])
    print(f"  Layer {layer}: {correct}/{len(layer_results)}")

save_path = f"/content/drive/MyDrive/Projects/PRISM/data/{model2b.cfg.model_name}_patching_eval_results_pass10_fan.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(gemma2b_results, f, indent=2, ensure_ascii=False)

print(f"\nAll results saved successfully to {save_path}")


3) Evaluating 2B/4B model with EACH layer patched individually (Zero patching)...
Downsampled dataset from 480 to 2240 samples for per-layer loop.


  0%|          | 0/2240 [00:00<?, ?it/s]

KeyboardInterrupt: 

# German

In [42]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"

In [45]:
def run_evaluations_for_language(lang: str):
    print(f"\n{'='*50}")
    print(f"=== Starting evaluation for {lang.upper()} ===")
    print(f"{'='*50}")

    # 1. Load dataset
    file_path = f"/content/drive/MyDrive/Projects/PRISM/data/rb_attrpatch_dataset_{lang}.json"
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            dataset = json.load(f)
        lang_data = dataset['pairs']
    except Exception as e:
        print(f"Could not load dataset for {lang}: {e}")
        return

    # --- 9B Evaluation ---
    """
    #print(f"\n[1] Evaluating 9B on {lang} (pass@10)...")
    #results_9b = []
    #for row in tqdm(lang_data, desc=f"9B {lang}"):
        system_text = row['system_rule']
        user_text = row['user_query']
        cat = row['category']
        grammar_type = row.get('grammar_type', 'unknown')

        input_ids = build_chat_tokens(system_text, user_text, 'hf', model_id).to(device)
        prompt_len = input_ids.shape[1]
        with torch.no_grad():
            outputs = model9b.generate(
                input_ids, max_new_tokens=60, temperature=0.7, do_sample=True, num_return_sequences=10, pad_token_id=tokenizer9b.eos_token_id
            )

        answers = []
        for i in range(outputs.shape[0]):
            gen_tokens = outputs[i, prompt_len:]
            ans = tokenizer9b.decode(gen_tokens, skip_special_tokens=True).strip()
            answers.append(ans)

        is_correct = False
        for ans in answers:
            try:
                if evaluate(row, ans):
                    is_correct = True
                    break
            except:
                pass

        results_9b.append({
            'rule_text': system_text, 'user_query': user_text, 'answer': answers, 'correct': is_correct,
            'rule_category': cat, 'rule_grammar_type': grammar_type,
            'model': model9b.config._name_or_path
       })

    save_name = model9b.config._name_or_path.replace('/', '_')
    save_path_9b = f"/content/drive/MyDrive/Projects/PRISM/data/{save_name}_evaluation_results_{lang}_pass10.json"
    with open(save_path_9b, "w", encoding="utf-8") as f:
        json.dump(results_9b, f, indent=2, ensure_ascii=False)
    print(f"9B results saved to {save_path_9b}")

    # --- 2B Evaluation ---
    print(f"\n[2] Evaluating 2B/4B on {lang} (pass@10)...")
    """
    model2b_results = []
    """
    print("  a) NO patching...")
    dummy_acts = torch.empty(0)
    for row in tqdm(lang_data, desc=f"2B No Patch {lang}"):
        system_text = row['system_rule']
        user_text = row['user_query']
        cat = row['category']
        grammar_type = row.get('grammar_type', 'unknown')

        input_ids = build_chat_tokens(system_text, user_text, 'tl', model2b).to(device)
        answers = generate_text_with_patching(model2b, input_ids, dummy_acts, [])

        is_correct = False
        for ans in answers:
            try:
                if evaluate(row, ans):
                    is_correct = True
                    break
            except:
                pass

        model2b_results.append({
            'rule_text': system_text, 'user_query': user_text, 'answer': answers, 'correct': is_correct,
            'rule_category': cat, 'rule_grammar_type': grammar_type,
            'model': model2b.cfg.model_name, 'patched_layers': []
        })

    print("  b) ALL layers patched (Zero patching)...")
    all_layers = list(range(n_layers2b))
    for row in tqdm(lang_data, desc=f"2B All Patch {lang}"):
        system_text = row['system_rule']
        user_text = row['user_query']
        cat = row['category']
        grammar_type = row.get('grammar_type', 'unknown')

        input_ids = build_chat_tokens(system_text, user_text, 'tl', model2b).to(device)
        prompt_len = input_ids.shape[-1]
        zero_acts = torch.zeros(n_layers2b, prompt_len, d_model2b, device=device, dtype=torch.float16)

        answers = generate_text_with_patching(model2b, input_ids, zero_acts, all_layers)

        is_correct = False
        for ans in answers:
            try:
                if evaluate(row, ans):
                    is_correct = True
                    break
            except:
                pass

        model2b_results.append({
            'rule_text': system_text, 'user_query': user_text, 'answer': answers, 'correct': is_correct,
            'rule_category': cat, 'rule_grammar_type': grammar_type,
            'model': model2b.cfg.model_name, 'patched_layers': all_layers
        })
    """
    print("  c) EACH layer patched individually (Zero patching)...")
    per_layer_lang_data = sample_dataset_by_category(lang_data, samples_per_category=10)
    print(f"  Downsampled {lang} dataset from {len(lang_data)} to {len(per_layer_lang_data)} samples for per-layer loop.")

    for row in tqdm(per_layer_lang_data, desc=f"2B Each Patch {lang}"):
        system_text = row['system_rule']
        user_text = row['user_query']
        cat = row['category']
        grammar_type = row.get('grammar_type', 'unknown')

        input_ids = build_chat_tokens(system_text, user_text, 'tl', model2b).to(device)
        prompt_len = input_ids.shape[-1]
        zero_acts = torch.zeros(n_layers2b, prompt_len, d_model2b, device=device, dtype=torch.float16)

        for layer in range(n_layers2b):
            answers = generate_text_with_patching(model2b, input_ids, zero_acts, [layer])

            is_correct = False
            for ans in answers:
                try:
                    if evaluate(row, ans):
                        is_correct = True
                        break
                except:
                    pass

            model2b_results.append({
                'rule_text': system_text, 'user_query': user_text, 'answer': answers, 'correct': is_correct,
                'rule_category': cat, 'rule_grammar_type': grammar_type,
                'model': model2b.cfg.model_name, 'patched_layers': [layer]
            })

    save_path_2b = f"/content/drive/MyDrive/Projects/PRISM/data/{model2b.cfg.model_name}_patching_eval_results_{lang}_pass10.json"
    with open(save_path_2b, "w", encoding="utf-8") as f:
        json.dump(model2b_results, f, indent=2, ensure_ascii=False)
    print(f"2B results saved to {save_path_2b}")


In [46]:
run_evaluations_for_language("de")


=== Starting evaluation for DE ===
  c) EACH layer patched individually (Zero patching)...
  Downsampled de dataset from 480 to 80 samples for per-layer loop.


2B Each Patch de:   0%|          | 0/80 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformer_lens/HookedTransformer.py:2058: UserWarning: HookedTransformer.generate received unsupported generation kwargs; ignoring: ['num_return_sequences', 'pad_token_id']
  warnings.warn(


2B results saved to /content/drive/MyDrive/Projects/PRISM/data/Llama-3.2-3B-Instruct_patching_eval_results_de_pass10.json


# Russian

In [48]:
run_evaluations_for_language("ru")


=== Starting evaluation for RU ===
  c) EACH layer patched individually (Zero patching)...
  Downsampled ru dataset from 480 to 80 samples for per-layer loop.


2B Each Patch ru:   0%|          | 0/80 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Italian

In [ ]:
run_evaluations_for_language("it")

# Cross-Language Layer Patching Visualization

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

model_name = model2b.cfg.model_name

# Map language names to their respective output JSON paths
# English was saved without a suffix in the previous steps
languages_and_paths = {
    "English": f"/content/drive/MyDrive/Projects/PRISM/data/{model_name}_patching_eval_results_pass10.json",
    "German": f"/content/drive/MyDrive/Projects/PRISM/data/{model_name}_patching_eval_results_de_pass10.json",
    "Russian": f"/content/drive/MyDrive/Projects/PRISM/data/{model_name}_patching_eval_results_ru_pass10.json",
    "Italian": f"/content/drive/MyDrive/Projects/PRISM/data/{model_name}_patching_eval_results_it_pass10.json"
}

data_records = []

for lang, path in languages_and_paths.items():
    if not os.path.exists(path):
        print(f"Warning: {path} not found. Has the {lang} evaluation finished?")
        continue

    try:
        with open(path, "r", encoding="utf-8") as f:
            results = json.load(f)
    except Exception as e:
        print(f"Error reading {lang} results: {e}")
        continue

    # Aggregate accuracy by layer
    layer_stats = {}
    for row in results:
        p_layers = row.get("patched_layers", [])

        # We only care about individually patched layers for this plot
        if len(p_layers) == 1:
            layer_idx = p_layers[0]
            is_correct = row.get("correct", False)

            if layer_idx not in layer_stats:
                layer_stats[layer_idx] = {"correct": 0, "total": 0}

            layer_stats[layer_idx]["total"] += 1
            if is_correct:
                layer_stats[layer_idx]["correct"] += 1

    # Calculate accuracy per layer and append to our dataset
    for layer_idx, stats in sorted(layer_stats.items()):
        accuracy = stats["correct"] / stats["total"] if stats["total"] > 0 else 0
        data_records.append({
            "Language": lang,
            "Layer": layer_idx,
            "Accuracy": accuracy
        })

if data_records:
    df = pd.DataFrame(data_records)

    # Plotting
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=df, x="Layer", y="Accuracy", hue="Language", marker="o", linewidth=2)

    plt.title(f"{model_name} Zero Patching Accuracy per Layer by Language (pass@10)", fontsize=14)
    plt.xlabel("Patched Layer Index", fontsize=12)
    plt.ylabel("Accuracy", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)

    # Make sure all layer ticks are shown on the x-axis
    max_layer = int(df["Layer"].max())
    plt.xticks(range(max_layer + 1))

    plt.legend(title="Language")
    plt.tight_layout()
    plt.show()
else:
    print("No valid data could be loaded. Please wait for the evaluations to finish saving.")
